In [ ]:
import os
os.environ["POLARS_FORCE_NEW_STREAMING"]="1"
import datetime
import colormaps
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs
import PIL

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"
os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
    to_expr,
    get_index_columns,
)
from jetutils.data import standardize, standardize_polars_dtypes, compute_anomalies_pl
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    slowness_from_cross,
    spells_from_cross,
    connected_from_cross,
    slowness_expr,
    spells_from_cross_catd_simple,
    extract_features,
    gaussian_smooth_func,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
    props_vs_quantiles,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import (
    create_bootstrapped_times,
    odds_ratio,
    is_signif_OR,
    common_OR,
)

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}
# pl.Config.set_verbose(True)
# pl.Config.set_streaming_chunk_size(500)  
basepath = Path(f"{DATADIR}/exp9")

ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)

In [ ]:
jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
phat_jets = to_one_large(jets)
props = pl.read_parquet(basepath.joinpath("props.parquet"))
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))
pers = slowness_from_cross(cross).drop("is_polar")
props = props.join(pers, on=["time", "jet ID"])

over_europe = pl.col("lon") > -10
lat_over_europe = (pl.col("lat") * pl.col("s")).filter(over_europe).sum() / pl.col(
    "s"
).filter(over_europe).sum()
lat_over_europe = jets.group_by("time", "jet ID").agg(
    lat_over_europe.fill_nan(0).alias("lat_over_europe")
)
props = props.join(lat_over_europe, on=["time", "jet ID"])

props_catd = squarify(average_jet_categories(props), ["time", "jet"])
props_catd = props_catd.join(
    props_catd.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
pers = props_catd.rolling("time", period="5d", group_by="jet").agg(
    pers=pl.col("slowness").sum()
)
props_catd = props_catd.join(pers, on=("time", "jet"))
props_catd_summer = summer_filter.join(props_catd, on="time")

dist_threshold = 2.0e6
overlap_threshold = 0.6
dis_polar_thresh = 0.12
spells_list = spells_from_cross(
    jets,
    cross,
    None,
    dist_threshold,
    overlap_threshold,
    dis_polar_thresh,
    n_STJ=30,
    n_EDJ=30,
    season=summer,
    STJ_lat_threshold=30,
)
_, summary_comp = connected_from_cross(
    jets,
    cross,
    dist_threshold,
    overlap_threshold,
    dis_polar_thresh,
)
jet_column = (
    pl.when((pl.col("is_polar") > 0.5).mean().over("spell") > 0.5)
    .then(pl.lit("EDJ"))
    .otherwise(pl.lit("STJ"))
)
summary_comp = (
    summary_comp.filter(pl.col("len") > 1)
    .drop("s", "theta", "is_polar", "s_right", "theta_right", "is_polar_right")
    .join(props["time", "jet ID", "is_polar"], on=("time", "jet ID"))
    .with_columns(
        jet=jet_column,
        slowness=slowness_expr(),
    )
    .with_columns(
        slowness_sum=pl.col("slowness").sum().over("spell"),
    )
    # .join(props, on=["time", "jet ID"], suffix="_fromprops")
    .sort("time", "jet ID")
)
summer_summary_comp = summary_comp.filter(
    pl.col("time")
    .is_in(pl.lit(summer.implode().first(), pl.List(pl.Datetime("ms"))))
    .over("spell")
)

for jet_name, spells in spells_list.items():
    print(jet_name, spells["spell"].n_unique())
    print(jet_name, spells["len"].min() / 4)

daily_spells_list = {
    a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()
}

In [ ]:
tf = pl.col("time").dt.year() == 1959
u = pl.scan_parquet(basepath.joinpath("u250_relative.parquet")).filter(tf).collect()
v = pl.scan_parquet(basepath.joinpath("v250_relative.parquet")).filter(tf).collect()
jets_ = jets.filter(tf)
uv = u.join(v.drop("is_polar"), on=["time", "jet ID", "norm_index", "n"])
uexpr = pl.col("u250_interp")
vexpr = pl.col("v250_interp")
sexpr = (uexpr.pow(2) + vexpr.pow(2)).sqrt()
uv = uv.with_columns(s250_interp=sexpr)
# uv = uv.join(uv.filter(pl.col("n") == 0).select("time", "jet ID", "norm_index", "s250_interp").rename({"s250_interp": "s_centre"}), on=["time", "jet ID", "norm_index"])
index = uv["norm_index"].unique().sort().to_frame().with_columns(index=pl.col("norm_index").rle_id())

In [ ]:
from jetutils.geospatial import haversine

period = 21
offset = int(np.ceil(period / 2))
# below = pl.col("s250_interp") <= pl.max_horizontal(pl.col("s_centre") / 4 * 3, pl.lit(10))
# stop_up = below.arg_max()
# n_up = pl.col("n").get(stop_up)
# n_up = pl.when(below.any()).then(n_up).otherwise(float("nan"))

# stop_down = below.len() - below.reverse().arg_max() - 1
# n_down = pl.col("n").get(stop_down)
# n_down = pl.when(below.any()).then(n_down).otherwise(float("nan"))

# half_width = (
#     pl.when(pl.col("side") < 0)
#     .then("n_down")
#     .otherwise("n_up")
# )
uv_smooth = (
    uv
    .with_columns(angle=pl.arctan2(vexpr, uexpr), side=pl.col("n").sign())
    # .group_by("time", "jet ID", "norm_index", side=pl.col("n").sign())
    # .agg(n_up=n_up, n_down=n_down)
    # .filter(pl.col("side") != 0)
    # .with_columns(half_width=half_width.fill_nan(None).abs())
    # .group_by("time", "jet ID", "norm_index")
    # .agg(width=pl.col("half_width").sum())
    # .drop("n_up", "n_down")
    # .sort("time", "jet ID", "norm_index", "side")
    .join(index, on="norm_index")
    # .with_columns(pl.col("s_").interpolate("nearest").fill_null(strategy="mean").over("side"))
    .rolling("index", group_by=["time", "jet ID", "n"], period=f"{period}i", offset=f"-{offset}i")
    .agg(pl.col("s250_interp").mean(), pl.col("angle").mean())
    .with_columns(
        u250_interp=pl.col("s250_interp") * pl.col("angle").cos(),
        v250_interp=pl.col("s250_interp") * pl.col("angle").sin()
    )
    .join(index, on="index")
    .drop("index")
)
# polars_to_xarray(uv.group_by(pl.col("is_polar") > 0.5, "n", "norm_index").agg(uexpr.mean(), vexpr.mean(), s250_interp=pl.col("s250_interp").mean()), ["is_polar", "n", "norm_index"])["s250_interp"][0].mean("norm_index").plot()

In [ ]:
uv_diff = uv.join(uv_smooth, on=["time", "jet ID", "n", "norm_index"]).with_columns(*[pl.col(f"{col}250_interp") - pl.col(f"{col}250_interp_right") for col in ["u", "v", "s"]]).drop(*[pl.col(f"{col}250_interp_right") for col in ["u", "v", "s"]]).with_columns(fake_eke=0.5 * (uexpr.pow(2) + vexpr.pow(2)))

In [ ]:
polars_to_xarray(uv_diff, ["time", "jet ID", "n", "norm_index"])["fake_eke"].mean(["time", "jet ID"]).plot()